#Init

In [0]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

#Read from Bronze Table

In [0]:
df = spark.table('workspace.bronze.erp_px_cat_g1v2')

#Data Transformations

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, T.StringType):
        trimmed_col = F.trim(F.col(field.name))
        
        df = df.withColumn(
            field.name, 
            F.when(trimmed_col == "", F.lit(None).cast("string"))
             .otherwise(trimmed_col)
        )

##Casting Maintenance into boolean

In [0]:
df = df.withColumn(
    "maintenance",
    F.when(F.upper(F.col("maintenance")) == "YES", F.lit(True))
     .when(F.upper(F.col("maintenance")) == "NO", F.lit(False))
     .otherwise(None)
)

## Renaming columns

In [0]:
RENAME_MAP = {
    "ID": "category_id",
    "CAT": "category",
    "SUBCAT": "subcategory",
    "MAINTENANCE": "requires_maintenance",
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

##Sanity checks Dataframe

In [0]:
df.limit(100).display()

#Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("workspace.silver.erp_product_category")

##Sanity checks Silver Table

In [0]:
%sql
SELECT * FROM silver.erp_product_category limit 10